# 02 — Annotation helper

1. **Sample** validation images → `validation_images.csv`
2. **Annotate** true SH lengths (mm) → `validation_lengths.csv` (or `validation_lengths2.csv`)

Run cells top to bottom. Re-run the last cell for each fish.


In [ ]:
import logging
import random
import sys
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from src.colab_bootstrap import setup_notebook_environment

REPO, STORAGE = setup_notebook_environment(mount_drive=True)
print(f"Storage: {STORAGE}")

from src.config import get_config
from src.dataset import build_split_index, discover_image_paths, iterate_dataset, load_sample
from src.masks import mask_from_class
from src.measurement import measure_bbox_length, measure_pca_length
from src.utils import setup_logging
from src.visualization import (
    display_sample,
    draw_polygon_annotations,
    overlay_masks,
    plot_pca_axis,
    save_figure,
)

setup_logging(level=logging.INFO)
cfg = get_config()

import pandas as pd

SPLIT = "valid"
N_SELECT = 30
SEED = 42

cfg.data_annotations.mkdir(parents=True, exist_ok=True)
images_csv = cfg.data_annotations / "validation_images.csv"
lengths_csv = cfg.data_annotations / "validation_lengths.csv"


In [ ]:
valid_paths = discover_image_paths(cfg, SPLIT)
print(f"Validation images available: {len(valid_paths)}")

rng = random.Random(SEED)
selected_paths = rng.sample(valid_paths, k=min(N_SELECT, len(valid_paths)))
selected_ids = [p.stem for p in sorted(selected_paths, key=lambda p: p.stem)]

pd.DataFrame({"image_id": selected_ids}).to_csv(images_csv, index=False)
print(f"Wrote {len(selected_ids)} IDs to {images_csv}")
selected_ids[:5], "..."


In [ ]:
# Load existing lengths if resuming
if lengths_csv.exists():
    lengths_df = pd.read_csv(lengths_csv)
    done = set(lengths_df["image_id"].astype(str))
    print(f"Resuming: {len(done)} already annotated")
else:
    lengths_df = pd.DataFrame(columns=["image_id", "true_length_mm"])
    done = set()

labels_dir = cfg.labels_dir(SPLIT)


In [ ]:
# Remaining validation IDs to annotate
pending_ids = pd.read_csv(images_csv)["image_id"].astype(str).tolist()
pending_ids = [i for i in pending_ids if i not in done]
print(f"Pending: {len(pending_ids)} / {len(pending_ids) + len(done)}")
pending_ids[:5]


In [ ]:
# Annotate one image at a time (re-run this cell for each fish)
SHOW_FISH_POLYGON = True

def annotate_next(length_mm: float) -> None:
    """Call after viewing the figure, e.g. annotate_next(59.0)."""
    if not pending_ids:
        print("All validation images annotated.")
        return
    image_id = pending_ids[0]
    sample_path = cfg.images_dir(SPLIT) / f"{image_id}.JPG"
    if not sample_path.is_file():
        sample_path = cfg.images_dir(SPLIT) / f"{image_id}.jpg"
    sample = load_sample(sample_path, split=SPLIT, load_image_array=True)
    fish_mask = mask_from_class(sample.annotations, "fish", sample.image.shape[0], sample.image.shape[1])

    fig, ax = plt.subplots(figsize=(11, 7))
    vis = draw_polygon_annotations(sample.image, sample.annotations) if SHOW_FISH_POLYGON else sample.image.copy()
    vis = overlay_masks(vis, {"fish": fish_mask})
    pca_len, center, axis = measure_pca_length(fish_mask)
    vis = plot_pca_axis(vis, center, axis, pca_len)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{image_id} — enter SH length (mm)")
    ax.axis("off")
    plt.show()

    global lengths_df, done, pending_ids
    row = pd.DataFrame([{"image_id": image_id, "true_length_mm": float(length_mm)}])
    lengths_df = pd.concat([lengths_df, row], ignore_index=True)
    lengths_df = lengths_df.drop_duplicates(subset=["image_id"], keep="last")
    lengths_df.to_csv(lengths_csv, index=False)
    done.add(image_id)
    pending_ids = [i for i in pending_ids if i != image_id]
    print(f"Saved {image_id} -> {length_mm} mm ({len(done)} total)")

# Example: annotate_next(59.0)
